# 03. Avaliar linkage

Pares do [`02b_aplicar_splink.ipynb`](02b_aplicar_splink.ipynb) (`p ≥ 0,5`).
Corte `T = THRESHOLD_AVALIACAO` (default 0,99).

Exemplos ≥ T e na faixa; melhor CPF; funil do Censo limpo (quem não saiu no
parquet mistura blocking e score `< 0,5`); discordância nome/DOB no melhor CPF;
ouro em cinco cortes; 1:1 abaixo de T. Resumo de contagens no fim.

Lista única no [`04_atribuir.ipynb`](04_atribuir.ipynb). A ouro é amostra 1:1,
não o universo.


In [ ]:
import sys
from pathlib import Path

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

from IPython.display import display

from config import (
    COHORT_DEDUP_ARQUIVO,
    SPLINK_INPUT_VIEW,
    SPLINK_PREDICTIONS,
    TABELA_CENSO_LIMPA,
    TABELA_CPF_LIMPA,
    THRESHOLD_AVALIACAO,
    drop_splink_temp_tables,
    get_connection,
    materialize_gt_no_subset,
    materialize_splink_input,
    print_paths,
    require_input,
    require_tables,
)

T = THRESHOLD_AVALIACAO
T_FAIXA = T - 0.05
TOP_N = 20

print_paths()
require_input(COHORT_DEDUP_ARQUIVO, label='COHORT')
require_input(SPLINK_PREDICTIONS, label='SPLINK_PREDICTIONS (rode o 02b_aplicar antes)')

con = get_connection()
drop_splink_temp_tables(con)
require_tables(con, [TABELA_CENSO_LIMPA, TABELA_CPF_LIMPA], notebook_origem='00b')
materialize_splink_input(con)
print('T:', T, '| faixa >=', T_FAIXA, 'e <', T)


Predictions no parquet (ids + score). Ouro 1:1 no recorte.


In [ ]:
cols_pred = list(
    con.execute(f"SELECT * FROM read_parquet('{SPLINK_PREDICTIONS}') LIMIT 0").df().columns
)
tem_weight = 'match_weight' in cols_pred
col_weight = ', match_weight' if tem_weight else ''
score_sql = 'p.match_probability'
if tem_weight:
    score_sql = 'p.match_probability, p.match_weight'
cols_exemplo = '''
    ca.nome_completo AS nome_censo,
    pb.nome_completo AS nome_cpf,
    ca.data_nascimento AS dob_censo,
    pb.data_nascimento AS dob_cpf,
    ca.idade AS idade_censo,
    pb.idade AS idade_cpf,
    ca.nome_mae AS mae_censo,
    pb.nome_mae AS mae_cpf,
    ca.sexo AS sexo_censo,
    pb.sexo AS sexo_cpf,
    ca.uf AS uf_censo,
    pb.uf AS uf_cpf,
    ca.cep AS cep_censo,
    pb.cep AS cep_cpf
'''

_tipo = con.execute('''
SELECT table_type
FROM information_schema.tables
WHERE table_schema = 'main' AND table_name = 'splink_predictions'
''').fetchone()
if _tipo:
    _kind = 'VIEW' if _tipo[0].upper() == 'VIEW' else 'TABLE'
    con.execute(f'DROP {_kind} IF EXISTS splink_predictions')
con.execute(f'''
CREATE OR REPLACE VIEW splink_predictions AS
SELECT
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_l
        ELSE unique_id_r
    END AS unique_id_censo,
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_r
        ELSE unique_id_l
    END AS unique_id_cpf,
    match_probability
    {col_weight}
FROM read_parquet('{SPLINK_PREDICTIONS}')
''')

n_pares = con.execute('SELECT COUNT(*) FROM splink_predictions').fetchone()[0]
print('Pares no parquet:', f'{n_pares:,}')

counts = materialize_gt_no_subset(con, cohort_parquet=COHORT_DEDUP_ARQUIVO)
n_gt = counts['n_gt_no_subset']
print('Ouro 1:1 no subset:', f'{n_gt:,}')
print('Não 1:1 descartados (N:1 / 1:N):', f"{counts['n_nao_1a1_descartada']:,}")
if n_gt == 0:
    raise RuntimeError(
        'Nenhum par da coorte caiu no subset — confira o filtro geográfico do NB00.'
    )


## Pares com p ≥ T

Candidatos acima do corte operacional.


In [ ]:
n_acima = con.execute(f'''
SELECT COUNT(*) FROM splink_predictions WHERE match_probability >= {T}
''').fetchone()[0]
print(f'Pares com p >= {T}:', f'{n_acima:,}')


In [ ]:
display(con.execute(f'''
SELECT
    p.unique_id_censo,
    p.unique_id_cpf,
    {score_sql},
    {cols_exemplo}
FROM splink_predictions p
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = p.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = p.unique_id_cpf
WHERE p.match_probability >= {T}
ORDER BY p.match_probability DESC, p.unique_id_censo
LIMIT {TOP_N}
''').df())


## Faixa [T − 0,05, T)

Pares perto do corte, ainda abaixo de T.


In [ ]:
n_faixa = con.execute(f'''
SELECT COUNT(*)
FROM splink_predictions
WHERE match_probability >= {T_FAIXA}
  AND match_probability < {T}
''').fetchone()[0]
print(f'Pares com {T_FAIXA} <= p < {T}:', f'{n_faixa:,}')


In [ ]:
display(con.execute(f'''
SELECT
    p.unique_id_censo,
    p.unique_id_cpf,
    {score_sql},
    {cols_exemplo}
FROM splink_predictions p
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = p.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = p.unique_id_cpf
WHERE p.match_probability >= {T_FAIXA}
  AND p.match_probability < {T}
ORDER BY p.match_probability DESC, p.unique_id_censo
LIMIT {TOP_N}
''').df())


## Melhor CPF por Censo (p ≥ T)

Cada Censo fica com o CPF de maior `match_probability` (empate: `unique_id_cpf`).
Não há greedy nem cluster. CPF com dois ou mais Censos no topo entra na conta
de múltiplos; associação única = esse CPF aparece uma vez.


In [ ]:
con.execute(f'''
CREATE OR REPLACE TABLE melhor_por_censo AS
SELECT unique_id_censo, unique_id_cpf, match_probability
FROM splink_predictions
WHERE match_probability >= {T}
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY unique_id_censo
    ORDER BY match_probability DESC, unique_id_cpf
) = 1
''')

con.execute('''
CREATE OR REPLACE TABLE cpf_n_censo AS
SELECT unique_id_cpf, COUNT(*) AS n_censo
FROM melhor_por_censo
GROUP BY 1
''')

con.execute('''
CREATE OR REPLACE TABLE associacoes_unicas AS
SELECT m.*
FROM melhor_por_censo m
JOIN cpf_n_censo c ON c.unique_id_cpf = m.unique_id_cpf
WHERE c.n_censo = 1
''')

display(con.execute(f'''
SELECT
    (SELECT COUNT(*) FROM splink_predictions WHERE match_probability >= {T})
        AS n_pares_acima_t,
    (SELECT COUNT(*) FROM melhor_por_censo) AS n_censos_com_par,
    (SELECT COUNT(*) FROM associacoes_unicas) AS n_associacoes_unicas,
    (SELECT COUNT(*) FROM cpf_n_censo WHERE n_censo = 1) AS n_cpf_unicos,
    (SELECT COUNT(*) FROM cpf_n_censo WHERE n_censo >= 2) AS n_cpf_multiplos
''').df())


In [ ]:
display(con.execute(f'''
SELECT
    m.unique_id_cpf,
    c.n_censo,
    m.unique_id_censo,
    m.match_probability,
    ca.nome_completo AS nome_censo,
    pb.nome_completo AS nome_cpf,
    ca.data_nascimento AS dob_censo,
    pb.data_nascimento AS dob_cpf,
    ca.idade AS idade_censo,
    pb.idade AS idade_cpf,
    ca.nome_mae AS mae_censo,
    pb.nome_mae AS mae_cpf
FROM melhor_por_censo m
JOIN cpf_n_censo c ON c.unique_id_cpf = m.unique_id_cpf
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = m.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = m.unique_id_cpf
WHERE c.n_censo >= 2
ORDER BY c.n_censo DESC, m.unique_id_cpf, m.match_probability DESC
LIMIT {TOP_N}
''').df())


## CPF compartilhado (p ≥ T)

Quantos Censos apontam para o mesmo melhor CPF (`n_censo ≥ 2`). Média por
CPF, não ponderada pelos Censos. Corte 3 é só leitura (duplicata possível no
Censo); `associacoes_unicas` continua `n_censo = 1`.


In [ ]:
import matplotlib.pyplot as plt

TETO_HIST = 15
CORTE_CPF = 3

display(con.execute('''
SELECT
    n_censo,
    COUNT(*) AS n_cpf,
    COUNT(*) * n_censo AS n_censo_linhas
FROM cpf_n_censo
WHERE n_censo >= 2
GROUP BY 1
ORDER BY 1
''').df())

display(con.execute(f'''
SELECT
    ROUND(AVG(n_censo), 2) AS media_n_censo_por_cpf,
    COUNT(*) AS n_cpf_compartilhado,
    COUNT(*) FILTER (WHERE n_censo <= {CORTE_CPF}) AS n_cpf_ate_3,
    COUNT(*) FILTER (WHERE n_censo > {CORTE_CPF}) AS n_cpf_mais_3,
    SUM(n_censo) AS n_censo_compartilhado,
    SUM(n_censo) FILTER (WHERE n_censo <= {CORTE_CPF}) AS n_censo_ate_3,
    SUM(n_censo) FILTER (WHERE n_censo > {CORTE_CPF}) AS n_censo_mais_3
FROM cpf_n_censo
WHERE n_censo >= 2
''').df())

plot = con.execute(f'''
SELECT
    CASE WHEN n_censo >= {TETO_HIST} THEN {TETO_HIST} ELSE n_censo END AS n_plot,
    COUNT(*) AS n_cpf
FROM cpf_n_censo
WHERE n_censo >= 2
GROUP BY 1
ORDER BY 1
''').df()

fig, ax = plt.subplots()
ax.bar(plot['n_plot'], plot['n_cpf'])
ax.axvline(CORTE_CPF + 0.5, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('Censos por CPF (melhor par, p ≥ T)')
ax.set_ylabel('Número de CPFs')
ax.set_title('CPF compartilhado: tamanho do grupo')
ticks = list(plot['n_plot'])
labels = [f'{TETO_HIST}+' if n == TETO_HIST else str(int(n)) for n in ticks]
ax.set_xticks(ticks)
ax.set_xticklabels(labels)
plt.show()


In [ ]:
display(con.execute(f'''
SELECT
    m.unique_id_cpf,
    c.n_censo,
    m.unique_id_censo,
    m.match_probability,
    ca.nome_completo AS nome_censo,
    pb.nome_completo AS nome_cpf,
    ca.nome_completo_phon AS nome_phon_censo,
    pb.nome_completo_phon AS nome_phon_cpf,
    ca.data_nascimento AS dob_censo,
    pb.data_nascimento AS dob_cpf,
    ca.idade AS idade_censo,
    pb.idade AS idade_cpf,
    ca.nome_mae AS mae_censo,
    pb.nome_mae AS mae_cpf
FROM melhor_por_censo m
JOIN cpf_n_censo c ON c.unique_id_cpf = m.unique_id_cpf
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = m.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = m.unique_id_cpf
WHERE c.n_censo IN (2, 3)
ORDER BY m.unique_id_cpf, m.unique_id_censo
''').df())


## Funil do Censo limpo

Universo: `origem = 'censo'` em `splink_input`. `n_sem_par_splink` não é fora da
blocagem — o parquet já é `p ≥ 0,5`.


In [ ]:
display(con.execute(f'''
SELECT
    n_censo,
    n_com_par_splink,
    n_censo - n_com_par_splink AS n_sem_par_splink,
    ROUND(100.0 * n_com_par_splink / NULLIF(n_censo, 0), 2) AS pct_com_par_splink,
    n_com_par_T,
    n_censo - n_com_par_T AS n_sem_par_T,
    ROUND(100.0 * n_com_par_T / NULLIF(n_censo, 0), 2) AS pct_com_par_T,
    n_associacoes_unicas,
    n_censo - n_associacoes_unicas AS n_sem_unica,
    ROUND(100.0 * n_associacoes_unicas / NULLIF(n_censo, 0), 2) AS pct_unica
FROM (
    SELECT
        (SELECT COUNT(*) FROM {SPLINK_INPUT_VIEW} WHERE origem = 'censo') AS n_censo,
        (SELECT COUNT(DISTINCT unique_id_censo) FROM splink_predictions) AS n_com_par_splink,
        (SELECT COUNT(*) FROM melhor_por_censo) AS n_com_par_T,
        (SELECT COUNT(*) FROM associacoes_unicas) AS n_associacoes_unicas
)
''').df())


In [ ]:
display(con.execute(f'''
SELECT
    s.unique_id,
    s.nome_completo,
    s.data_nascimento,
    s.sexo,
    s.uf,
    s.cep
FROM {SPLINK_INPUT_VIEW} s
LEFT JOIN (
    SELECT DISTINCT unique_id_censo FROM splink_predictions
) p ON p.unique_id_censo = s.unique_id
WHERE s.origem = 'censo'
  AND p.unique_id_censo IS NULL
LIMIT {TOP_N}
''').df())


## Discordância no melhor CPF

Nome fonético (`nome_completo_phon`) ou DOB preenchidos e diferentes. Grafia
que a fonética igualou não conta. Nulo não conta. Só `melhor_por_censo`.


In [ ]:
display(con.execute(f'''
SELECT
    COUNT(*) AS n_discorda,
    COUNT(*) FILTER (
        WHERE ca.nome_completo_phon IS NOT NULL
          AND pb.nome_completo_phon IS NOT NULL
          AND ca.nome_completo_phon <> pb.nome_completo_phon
    ) AS n_nome_discorda,
    COUNT(*) FILTER (
        WHERE ca.data_nascimento IS NOT NULL
          AND pb.data_nascimento IS NOT NULL
          AND ca.data_nascimento <> pb.data_nascimento
    ) AS n_dob_discorda
FROM melhor_por_censo m
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = m.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = m.unique_id_cpf
WHERE (
        ca.nome_completo_phon IS NOT NULL
        AND pb.nome_completo_phon IS NOT NULL
        AND ca.nome_completo_phon <> pb.nome_completo_phon
    )
    OR (
        ca.data_nascimento IS NOT NULL
        AND pb.data_nascimento IS NOT NULL
        AND ca.data_nascimento <> pb.data_nascimento
    )
''').df())

display(con.execute(f'''
SELECT
    m.unique_id_censo,
    m.unique_id_cpf,
    m.match_probability,
    (ca.nome_completo_phon IS NOT NULL AND pb.nome_completo_phon IS NOT NULL
     AND ca.nome_completo_phon <> pb.nome_completo_phon) AS nome_discorda,
    (ca.data_nascimento IS NOT NULL AND pb.data_nascimento IS NOT NULL
     AND ca.data_nascimento <> pb.data_nascimento) AS dob_discorda,
    ca.nome_completo_phon AS nome_phon_censo,
    pb.nome_completo_phon AS nome_phon_cpf,
    {cols_exemplo}
FROM melhor_por_censo m
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = m.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = m.unique_id_cpf
WHERE (
        ca.nome_completo_phon IS NOT NULL
        AND pb.nome_completo_phon IS NOT NULL
        AND ca.nome_completo_phon <> pb.nome_completo_phon
    )
    OR (
        ca.data_nascimento IS NOT NULL
        AND pb.data_nascimento IS NOT NULL
        AND ca.data_nascimento <> pb.data_nascimento
    )
ORDER BY m.match_probability DESC, m.unique_id_censo
LIMIT {TOP_N}
''').df())


## Ouro (cinco cortes)

Amostra 1:1 da coorte, não o universo. Cada Censo ouro cai em um corte.
`unica_errada` é o erro operacional visível; compartilhada / abaixo_T /
sem_par_splink não são FP.


In [ ]:
display(con.execute('''
SELECT
    COUNT(*) AS n_ouro,
    COUNT(*) FILTER (
        WHERE u.unique_id_cpf = gt.unique_id_cpf
    ) AS n_encontrada,
    COUNT(*) FILTER (
        WHERE u.unique_id_censo IS NOT NULL
          AND u.unique_id_cpf <> gt.unique_id_cpf
    ) AS n_unica_errada,
    COUNT(*) FILTER (
        WHERE m.unique_id_censo IS NOT NULL
          AND u.unique_id_censo IS NULL
    ) AS n_compartilhada,
    COUNT(*) FILTER (
        WHERE p.unique_id_censo IS NOT NULL
          AND m.unique_id_censo IS NULL
    ) AS n_abaixo_T,
    COUNT(*) FILTER (
        WHERE p.unique_id_censo IS NULL
    ) AS n_sem_par_splink,
    ROUND(
        100.0 * COUNT(*) FILTER (WHERE u.unique_id_cpf = gt.unique_id_cpf)
        / NULLIF(COUNT(*), 0),
        2
    ) AS recall_pct,
    COUNT(DISTINCT CASE
        WHEN u_cpf.unique_id_cpf IS NOT NULL THEN gt.unique_id_cpf
    END) AS n_cpf_ouro_nas_unicas
FROM gt_no_subset gt
LEFT JOIN associacoes_unicas u ON u.unique_id_censo = gt.unique_id_censo
LEFT JOIN melhor_por_censo m ON m.unique_id_censo = gt.unique_id_censo
LEFT JOIN (
    SELECT DISTINCT unique_id_censo FROM splink_predictions
) p ON p.unique_id_censo = gt.unique_id_censo
LEFT JOIN associacoes_unicas u_cpf ON u_cpf.unique_id_cpf = gt.unique_id_cpf
''').df())


In [ ]:
display(con.execute(f'''
SELECT
    u.unique_id_censo,
    u.unique_id_cpf,
    gt.unique_id_cpf AS unique_id_cpf_ouro,
    u.match_probability,
    {cols_exemplo}
FROM gt_no_subset gt
JOIN associacoes_unicas u ON u.unique_id_censo = gt.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = u.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = u.unique_id_cpf
WHERE u.unique_id_cpf <> gt.unique_id_cpf
ORDER BY u.match_probability DESC, u.unique_id_censo
LIMIT {TOP_N}
''').df())


## 1:1 abaixo de T

No recorte `p < T` (o parquet já é `p ≥ 0,5`): Censo com exatamente um CPF e
esse CPF com exatamente um Censo. Sem escolher o melhor score.


In [ ]:
con.execute(f'''
CREATE OR REPLACE TABLE um_para_um_abaixo AS
SELECT unique_id_censo, unique_id_cpf, match_probability
FROM (
    SELECT
        unique_id_censo,
        unique_id_cpf,
        match_probability,
        COUNT(*) OVER (PARTITION BY unique_id_censo) AS n_cpf,
        COUNT(*) OVER (PARTITION BY unique_id_cpf) AS n_censo
    FROM splink_predictions
    WHERE match_probability < {T}
)
WHERE n_cpf = 1 AND n_censo = 1
''')

n_1a1 = con.execute('SELECT COUNT(*) FROM um_para_um_abaixo').fetchone()[0]
print(f'Pares 1:1 com p < {T}:', f'{n_1a1:,}')


In [ ]:
display(con.execute(f'''
SELECT
    p.unique_id_censo,
    p.unique_id_cpf,
    p.match_probability,
    {cols_exemplo}
FROM um_para_um_abaixo p
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = p.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = p.unique_id_cpf
ORDER BY p.match_probability DESC, p.unique_id_censo
LIMIT {TOP_N}
''').df())


## Resumo

Censos com par `p ≥ T`; associações únicas (1 Censo : 1 CPF); pares 1:1 na
faixa `[0,90, T)` (Censo com um CPF **nessa faixa** e esse CPF com um Censo);
únicas em que `nome_completo_phon` e `data_nascimento` **não** são iguais nos
dois lados (nulo não conta como igual). `n_unica_com_nome_dob` +
`n_unica_sem_nome_dob` = `n_associacoes_unicas`.


In [ ]:
T_90 = 0.90

display(con.execute(f'''
SELECT
    (SELECT COUNT(*) FROM melhor_por_censo) AS n_censo_com_par_T,
    (SELECT COUNT(*) FROM associacoes_unicas) AS n_associacoes_unicas,
    (
        SELECT COUNT(*)
        FROM (
            SELECT unique_id_censo, unique_id_cpf,
                COUNT(*) OVER (PARTITION BY unique_id_censo) AS n_cpf,
                COUNT(*) OVER (PARTITION BY unique_id_cpf) AS n_censo
            FROM splink_predictions
            WHERE match_probability >= {T_90}
              AND match_probability < {T}
        )
        WHERE n_cpf = 1 AND n_censo = 1
    ) AS n_1a1_090_T,
    COUNT(*) FILTER (
        WHERE ca.nome_completo_phon = pb.nome_completo_phon
          AND ca.data_nascimento = pb.data_nascimento
    ) AS n_unica_com_nome_dob,
    COUNT(*) - COUNT(*) FILTER (
        WHERE ca.nome_completo_phon = pb.nome_completo_phon
          AND ca.data_nascimento = pb.data_nascimento
    ) AS n_unica_sem_nome_dob
FROM associacoes_unicas u
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = u.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = u.unique_id_cpf
''').df())


Mudar `T_RESUMO` e rerodar só esta célula. Recalcula o resumo a partir do
parquet (`p ≥ 0,5`); não altera `melhor_por_censo` nem o `T` do setup.
Cortes abaixo de 0,5 não criam par novo.


In [ ]:
T_RESUMO = 0.99
T_90 = 0.90

display(con.execute(f'''
WITH melhor AS (
    SELECT unique_id_censo, unique_id_cpf, match_probability
    FROM splink_predictions
    WHERE match_probability >= {T_RESUMO}
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY unique_id_censo
        ORDER BY match_probability DESC, unique_id_cpf
    ) = 1
),
cpf_n AS (
    SELECT unique_id_cpf, COUNT(*) AS n_censo
    FROM melhor
    GROUP BY 1
),
unicas AS (
    SELECT m.*
    FROM melhor m
    JOIN cpf_n c ON c.unique_id_cpf = m.unique_id_cpf
    WHERE c.n_censo = 1
)
SELECT
    {T_RESUMO} AS T_RESUMO,
    (SELECT COUNT(*) FROM melhor) AS n_censo_com_par_T,
    (SELECT COUNT(*) FROM unicas) AS n_associacoes_unicas,
    (
        SELECT COUNT(*)
        FROM (
            SELECT unique_id_censo, unique_id_cpf,
                COUNT(*) OVER (PARTITION BY unique_id_censo) AS n_cpf,
                COUNT(*) OVER (PARTITION BY unique_id_cpf) AS n_censo
            FROM splink_predictions
            WHERE match_probability >= {T_90}
              AND match_probability < {T_RESUMO}
        )
        WHERE n_cpf = 1 AND n_censo = 1
    ) AS n_1a1_090_T,
    COUNT(*) FILTER (
        WHERE ca.nome_completo_phon = pb.nome_completo_phon
          AND ca.data_nascimento = pb.data_nascimento
    ) AS n_unica_com_nome_dob,
    COUNT(*) - COUNT(*) FILTER (
        WHERE ca.nome_completo_phon = pb.nome_completo_phon
          AND ca.data_nascimento = pb.data_nascimento
    ) AS n_unica_sem_nome_dob
FROM unicas u
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = u.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = u.unique_id_cpf
''').df())


In [ ]:
con.close()
